In [3]:
from CCXTDataFetcher import CCXTDataFetcher
from datetime import datetime

# Initialize the fetcher
fetcher = CCXTDataFetcher()

# Define different date ranges
spot_start_date = datetime(2024, 6, 1)      # June 2024 for spot
futures_start_date = datetime(2025, 1, 1)   # January 2025 for futures
end_date = datetime.now()                    # Up to now

print(f"Spot data will be fetched from: {spot_start_date.strftime('%Y-%m-%d')}")
print(f"Futures data will be fetched from: {futures_start_date.strftime('%Y-%m-%d')}")
print(f"End date: {end_date.strftime('%Y-%m-%d')}")

Spot data will be fetched from: 2024-06-01
Futures data will be fetched from: 2025-01-01
End date: 2025-09-01


In [2]:
# Define tokens for spot data
spot_tokens = ['ETH', 'MORPHO', 'INJ', 'LINK', 'WTAO']

print("Fetching SPOT data from Binance (June 2024 - Now):")
print("=" * 60)

spot_results = {}
for token in spot_tokens:
    if token != 'USDC':  # Skip USDC as it's stable
        result = fetcher.fetch_spot_data(
            symbol=token,
            timeframe='5m',
            start_date=spot_start_date,
            end_date=end_date
        )
        if result is not None:
            spot_results[token] = result
        print()  # Empty line for readability

Fetching SPOT data from Binance (June 2024 - Now):
Fetching 5m spot data for ETH from Binance...
Date range: 2024-06-01 to 2025-08-30
✓ Downloaded 131229 records for ETH
  Saved to: ./data/centralized_prices/spot/eth_binance_5m.csv
  Date range: 2024-05-31 22:00:00 to 2025-08-30 13:40:00

Fetching 5m spot data for MORPHO from Binance...
Date range: 2024-06-01 to 2025-08-30
✓ Downloaded 79479 records for MORPHO
  Saved to: ./data/centralized_prices/spot/morpho_binance_5m.csv
  Date range: 2024-11-27 14:30:00 to 2025-08-30 13:40:00

Fetching 5m spot data for INJ from Binance...
Date range: 2024-06-01 to 2025-08-30
✓ Downloaded 131229 records for INJ
  Saved to: ./data/centralized_prices/spot/inj_binance_5m.csv
  Date range: 2024-05-31 22:00:00 to 2025-08-30 13:40:00

Fetching 5m spot data for LINK from Binance...
Date range: 2024-06-01 to 2025-08-30
✓ Downloaded 131230 records for LINK
  Saved to: ./data/centralized_prices/spot/link_binance_5m.csv
  Date range: 2024-05-31 22:00:00 to 202

In [5]:
# Define tokens for futures data
futures_tokens = ['ETH', 'MORPHO', 'INJ', 'LINK', 'WTAO']

print("Fetching FUTURES data from Bitget (January 2025 - Now):")
print("=" * 60)

futures_results = {}
for token in futures_tokens:
    if token != 'USDC':  # Skip USDC as it's stable
        result = fetcher.fetch_futures_data(
            symbol=token,
            timeframe='1h',
            start_date=futures_start_date,
            end_date=end_date
        )
        if result is not None:
            futures_results[token] = result
        print()  # Empty line for readability

Fetching FUTURES data from Bitget (January 2025 - Now):
Fetching 1h futures data for ETH from Bitget...
Date range: 2025-01-01 to 2025-09-01
✓ Downloaded 800 records for ETH
  Saved to: ./data/centralized_prices/futures/eth_bitget_futures_1h.csv
  Date range: 2025-02-03 07:00:00 to 2025-06-16 11:00:00

Fetching 1h futures data for MORPHO from Bitget...
Date range: 2025-01-01 to 2025-09-01
✓ Downloaded 800 records for MORPHO
  Saved to: ./data/centralized_prices/futures/morpho_bitget_futures_1h.csv
  Date range: 2025-02-03 07:00:00 to 2025-06-16 11:00:00

Fetching 1h futures data for INJ from Bitget...
Date range: 2025-01-01 to 2025-09-01
✓ Downloaded 800 records for INJ
  Saved to: ./data/centralized_prices/futures/inj_bitget_futures_1h.csv
  Date range: 2025-02-03 07:00:00 to 2025-06-16 11:00:00

Fetching 1h futures data for LINK from Bitget...
Date range: 2025-01-01 to 2025-09-01
✓ Downloaded 800 records for LINK
  Saved to: ./data/centralized_prices/futures/link_bitget_futures_1h.cs

In [4]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import os
import glob
import warnings
warnings.filterwarnings('ignore')

class SpotVolatilityCalculator:
    """
    Calculates and adds volatility columns to spot price data
    """
    
    def __init__(self, lookback_days=90, half_life_days=30):
        """
        Args:
            lookback_days: Total lookback period (default: 90 days = 3 months)
            half_life_days: Half-life for exponential weighting (default: 30 days = 1 month)
        """
        self.lookback_days = lookback_days
        self.half_life_days = half_life_days
        
        # Calculate decay factor for exponential weighting
        self.decay_factor = np.log(2) / half_life_days
        
        print(f"Spot Volatility Calculator initialized:")
        print(f"  Lookback period: {lookback_days} days")
        print(f"  Half-life: {half_life_days} days")
        print(f"  Decay factor: {self.decay_factor:.6f}")
    
    def calculate_rolling_volatility(self, price_series, frequency='5m'):
        """
        Calculate rolling volatility with exponential weighting
        
        Args:
            price_series: Series of close prices
            frequency: Data frequency ('5m', 'H', 'D')
        
        Returns:
            Series with rolling volatility
        """
        # Calculate returns
        returns = price_series.pct_change().dropna()
        
        # Calculate lookback periods based on frequency
        if frequency == '5m':
            lookback_periods = self.lookback_days * 24 * 12  # 12 5-min periods per hour
        elif frequency == 'H':
            lookback_periods = self.lookback_days * 24  # 24 hours per day
        elif frequency == 'D':
            lookback_periods = self.lookback_days
        else:
            # Default to 5-minute
            lookback_periods = self.lookback_days * 24 * 12
        
        # Initialize volatility column
        volatility = pd.Series(index=returns.index, dtype=float)
        
        # Calculate rolling volatility with exponential weighting
        for i in range(len(returns)):
            if i < lookback_periods:
                # Not enough data yet, use simple volatility
                volatility.iloc[i] = returns.iloc[:i+1].std()
            else:
                # Calculate exponentially weighted volatility
                start_idx = i - lookback_periods + 1
                window_returns = returns.iloc[start_idx:i+1]
                
                # Create exponential weights (more weight to recent data)
                weights = np.exp(-self.decay_factor * np.arange(len(window_returns))[::-1])
                weights = weights / weights.sum()  # Normalize weights
                
                # Calculate weighted volatility
                weighted_mean = np.average(window_returns, weights=weights)
                weighted_variance = np.average((window_returns - weighted_mean)**2, weights=weights)
                volatility.iloc[i] = np.sqrt(weighted_variance)
        
        # Annualize volatility based on frequency
        if frequency == '5m':
            annualization_factor = np.sqrt(12 * 24 * 365)  # 5-min to annual
        elif frequency == 'H':
            annualization_factor = np.sqrt(24 * 365)  # Hourly to annual
        elif frequency == 'D':
            annualization_factor = np.sqrt(365)  # Daily to annual
        else:
            annualization_factor = np.sqrt(12 * 24 * 365)  # Default to 5-min
        
        volatility_annualized = volatility * annualization_factor
        
        return volatility_annualized
    
    def add_volatility_to_spot_data(self, data_dir='./data/centralized_prices'):
        """
        Add volatility columns to all spot price CSV files
        """
        spot_dir = os.path.join(data_dir, 'spot')
        futures_dir = os.path.join(data_dir, 'futures')
        
        print("Adding volatility columns to spot price data...")
        print("=" * 60)
        
        # Process spot data
        if os.path.exists(spot_dir):
            print(f"\nProcessing SPOT data in: {spot_dir}")
            self._process_directory(spot_dir, 'spot')
        
        # Process futures data
        if os.path.exists(futures_dir):
            print(f"\nProcessing FUTURES data in: {futures_dir}")
            self._process_directory(futures_dir, 'futures')
        
        print("\n" + "=" * 60)
        print("Volatility calculation complete!")
    
    def _process_directory(self, directory, data_type):
        """Process all CSV files in a directory"""
        csv_files = glob.glob(os.path.join(directory, '*_binance_5m.csv'))
        
        if data_type == 'futures':
            csv_files.extend(glob.glob(os.path.join(directory, '*_bitget_futures_5m.csv')))
        
        print(f"Found {len(csv_files)} CSV files to process")
        
        for file in csv_files:
            self._process_single_file(file, data_type)
    
    def _process_single_file(self, file_path, data_type):
        """Process a single CSV file and add volatility columns"""
        filename = os.path.basename(file_path)
        print(f"\nProcessing: {filename}")
        
        try:
            # Load data
            data = pd.read_csv(file_path, index_col='datetime', parse_dates=True)
            
            if 'close' not in data.columns:
                print(f"  ❌ No 'close' column found, skipping...")
                return
            
            print(f"  Original shape: {data.shape}")
            print(f"  Date range: {data.index.min()} to {data.index.max()}")
            
            # Determine frequency from filename
            if '5m' in filename:
                frequency = '5m'
            elif '1h' in filename or '1H' in filename:
                frequency = 'H'
            elif '1d' in filename or '1D' in filename:
                frequency = 'D'
            else:
                frequency = '5m'  # Default
            
            # Calculate volatility
            print(f"  Calculating volatility (frequency: {frequency})...")
            volatility = self.calculate_rolling_volatility(data['close'], frequency)
            
            # Add volatility columns
            data['volatility'] = volatility
            data['volatility_pct'] = volatility * 100  # As percentage
            
            # Calculate additional metrics
            data['returns'] = data['close'].pct_change()
            data['log_returns'] = np.log(data['close'] / data['close'].shift(1))
            
            # Calculate rolling statistics
            lookback_periods = self.lookback_days * 24 * 12 if frequency == '5m' else self.lookback_days * 24
            data['rolling_mean'] = data['close'].rolling(window=lookback_periods).mean()
            data['rolling_std'] = data['close'].rolling(window=lookback_periods).std()
            
            # Save updated data (overwrite original file)
            data.to_csv(file_path)
            
            print(f"  ✓ Added volatility columns successfully")
            print(f"  Updated shape: {data.shape}")
            print(f"  Volatility range: {volatility.min():.2%} - {volatility.max():.2%}")
            print(f"  Saved to: {file_path}")
            
        except Exception as e:
            print(f"  ❌ Error processing {filename}: {e}")
            import traceback
            traceback.print_exc()

In [5]:
# Initialize calculator
vol_calculator = SpotVolatilityCalculator(lookback_days=90, half_life_days=30)

# Add volatility to all spot price data
vol_calculator.add_volatility_to_spot_data()

Spot Volatility Calculator initialized:
  Lookback period: 90 days
  Half-life: 30 days
  Decay factor: 0.023105
Adding volatility columns to spot price data...

Processing SPOT data in: ./data/centralized_prices/spot
Found 5 CSV files to process

Processing: link_binance_5m.csv
  Original shape: (131230, 6)
  Date range: 2024-05-31 22:00:00 to 2025-08-30 13:45:00
  Calculating volatility (frequency: 5m)...
  ✓ Added volatility columns successfully
  Updated shape: (131230, 12)
  Volatility range: 27.43% - 617.51%
  Saved to: ./data/centralized_prices/spot/link_binance_5m.csv

Processing: inj_binance_5m.csv
  Original shape: (131229, 6)
  Date range: 2024-05-31 22:00:00 to 2025-08-30 13:40:00
  Calculating volatility (frequency: 5m)...
  ✓ Added volatility columns successfully
  Updated shape: (131229, 12)
  Volatility range: 18.70% - 710.16%
  Saved to: ./data/centralized_prices/spot/inj_binance_5m.csv

Processing: morpho_binance_5m.csv
  Original shape: (79479, 6)
  Date range: 2024-

KeyboardInterrupt: 

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import os
import glob
import warnings
warnings.filterwarnings('ignore')

class HourlyVolatilityAdder:
    """
    Adds hourly volatility to existing 5-minute data files
    Resamples 5-min data to hourly, calculates volatility, then forward fills
    """
    
    def __init__(self, lookback_days=90, half_life_days=30):
        """
        Args:
            lookback_days: Total lookback period (default: 90 days = 3 months)
            half_life_days: Half-life for exponential weighting (default: 30 days = 1 month)
        """
        self.lookback_days = lookback_days
        self.half_life_days = half_life_days
        
        # Calculate decay factor for exponential weighting
        self.decay_factor = np.log(2) / half_life_days
        
        print(f"Hourly Volatility Adder initialized:")
        print(f"  Lookback period: {lookback_days} days")
        print(f"  Half-life: {half_life_days} days")
        print(f"  Decay factor: {self.decay_factor:.6f}")
    
    def calculate_hourly_volatility(self, price_series):
        """
        Calculate hourly volatility from hourly price data
        """
        # Calculate returns
        returns = price_series.pct_change().dropna()
        
        # Calculate lookback periods (90 days * 24 hours)
        lookback_periods = self.lookback_days * 24
        
        # Initialize volatility column
        volatility = pd.Series(index=returns.index, dtype=float)
        
        # Calculate rolling volatility with exponential weighting
        for i in range(len(returns)):
            if i < lookback_periods:
                # Not enough data yet, use simple volatility
                volatility.iloc[i] = returns.iloc[:i+1].std()
            else:
                # Calculate exponentially weighted volatility
                start_idx = i - lookback_periods + 1
                window_returns = returns.iloc[start_idx:i+1]
                
                # Create exponential weights (more weight to recent data)
                weights = np.exp(-self.decay_factor * np.arange(len(window_returns))[::-1])
                weights = weights / weights.sum()  # Normalize weights
                
                # Calculate weighted volatility
                weighted_mean = np.average(window_returns, weights=weights)
                weighted_variance = np.average((window_returns - weighted_mean)**2, weights=weights)
                volatility.iloc[i] = np.sqrt(weighted_variance)
        
        # Annualize volatility (hourly to annual)
        annualization_factor = np.sqrt(24 * 365)
        volatility_annualized = volatility * annualization_factor
        
        return volatility_annualized
    
    def add_hourly_volatility_to_files(self, data_dir='./data/centralized_prices'):
        """
        Add hourly volatility to all existing 5-minute data files
        """
        spot_dir = os.path.join(data_dir, 'spot')
        futures_dir = os.path.join(data_dir, 'futures')
        
        print("Adding hourly volatility to existing 5-minute data files...")
        print("=" * 60)
        
        # Process spot data
        if os.path.exists(spot_dir):
            print(f"\nProcessing SPOT data in: {spot_dir}")
            self._process_directory(spot_dir, 'spot')
        
        # Process futures data
        if os.path.exists(futures_dir):
            print(f"\nProcessing FUTURES data in: {futures_dir}")
            self._process_directory(futures_dir, 'futures')
        
        print("\n" + "=" * 60)
        print("Hourly volatility addition complete!")
    
    def _process_directory(self, directory, data_type):
        """Process all CSV files in a directory"""
        if data_type == 'spot':
            csv_files = glob.glob(os.path.join(directory, '*_binance_5m.csv'))
        else:  # futures
            csv_files = glob.glob(os.path.join(directory, '*_bitget_futures_5m.csv'))
        
        print(f"Found {len(csv_files)} CSV files to process")
        
        for file in csv_files:
            self._process_single_file(file, data_type)
    
    def _process_single_file(self, file_path, data_type):
        """Process a single CSV file and add hourly volatility"""
        filename = os.path.basename(file_path)
        print(f"\nProcessing: {filename}")
        
        try:
            # Load existing data
            data = pd.read_csv(file_path, index_col='datetime', parse_dates=True)
            
            if 'close' not in data.columns:
                print(f"  ❌ No 'close' column found, skipping...")
                return
            
            print(f"  Original shape: {data.shape}")
            print(f"  Date range: {data.index.min()} to {data.index.max()}")
            
            # Check if hourly volatility already exists
            if 'volatility_hourly' in data.columns:
                print(f"  ⚠️  Hourly volatility already exists, skipping...")
                return
            
            # Resample 5-minute data to hourly
            print(f"  Resampling 5-minute data to hourly...")
            data_hourly = data.resample('H').agg({
                'open': 'first',
                'high': 'max',
                'low': 'min',
                'close': 'last',
                'volume': 'sum'
            }).dropna()
            
            print(f"  Hourly data shape: {data_hourly.shape}")
            
            # Calculate hourly volatility
            print(f"  Calculating hourly volatility...")
            hourly_vol = self.calculate_hourly_volatility(data_hourly['close'])
            
            # Add hourly volatility to hourly data
            data_hourly['volatility_hourly'] = hourly_vol
            data_hourly['volatility_hourly_pct'] = hourly_vol * 100
            
            # Forward fill hourly volatility to 5-minute periods
            print(f"  Forward filling hourly volatility to 5-minute periods...")
            
            # Create a mapping from hourly to 5-minute timestamps
            hourly_vol_mapping = data_hourly['volatility_hourly']
            
            # Forward fill the hourly volatility to all 5-minute periods
            data['volatility_hourly'] = hourly_vol_mapping.reindex(data.index, method='ffill')
            data['volatility_hourly_pct'] = data['volatility_hourly'] * 100
            
            # Also add the hourly timestamp for reference
            data['hour_timestamp'] = data.index.floor('H')
            
            # Save updated data (overwrite original file)
            data.to_csv(file_path)
            
            print(f"  ✓ Added hourly volatility columns successfully")
            print(f"  Updated shape: {data.shape}")
            print(f"  Hourly volatility range: {hourly_vol.min():.2%} - {hourly_vol.max():.2%}")
            print(f"  Columns added: volatility_hourly, volatility_hourly_pct, hour_timestamp")
            print(f"  Saved to: {file_path}")
            
        except Exception as e:
            print(f"  ❌ Error processing {filename}: {e}")
            import traceback
            traceback.print_exc()

In [ ]:
# Initialize the hourly volatility adder
hourly_vol_adder = HourlyVolatilityAdder(lookback_days=90, half_life_days=30)

# Add hourly volatility to all existing files
hourly_vol_adder.add_hourly_volatility_to_files()

Hourly Volatility Adder initialized:
  Lookback period: 90 days
  Half-life: 30 days
  Decay factor: 0.023105
Adding hourly volatility to existing 5-minute data files...

Processing SPOT data in: ./data/centralized_prices/spot
Found 5 CSV files to process

Processing: link_binance_5m.csv
  Original shape: (131230, 12)
  Date range: 2024-05-31 22:00:00 to 2025-08-30 13:45:00
  Resampling 5-minute data to hourly...
  Hourly data shape: (10936, 5)
  Calculating hourly volatility...
  Forward filling hourly volatility to 5-minute periods...
  ✓ Added hourly volatility columns successfully
  Updated shape: (131230, 15)
  Hourly volatility range: 19.17% - 246.98%
  Columns added: volatility_hourly, volatility_hourly_pct, hour_timestamp
  Saved to: ./data/centralized_prices/spot/link_binance_5m.csv

Processing: inj_binance_5m.csv
  Original shape: (131229, 12)
  Date range: 2024-05-31 22:00:00 to 2025-08-30 13:40:00
  Resampling 5-minute data to hourly...
  Hourly data shape: (10936, 5)
  Cal

In [3]:
# Check what token names are actually available in your spot data
def check_available_tokens():
    """Check what tokens are actually available in the spot data"""
    spot_dir = './data/centralized_prices/spot'
    
    if os.path.exists(spot_dir):
        csv_files = glob.glob(os.path.join(spot_dir, '*_binance_5m.csv'))
        
        print("Available token files:")
        print("=" * 40)
        
        available_tokens = []
        for file in csv_files:
            token_name = os.path.basename(file).split('_')[0].upper()
            available_tokens.append(token_name)
            print(f"  {token_name}")
        
        print(f"\nTotal tokens found: {len(available_tokens)}")
        return available_tokens
    else:
        print("Spot data directory not found")
        return []

# Check what's available
available_tokens = check_available_tokens()

Available token files:
  LINK
  INJ
  MORPHO
  ETH
  WTAO

Total tokens found: 5


In [4]:
# Update the pool configuration based on what's actually available
def get_corrected_pool_config(available_tokens):
    """
    Get corrected pool configuration based on available tokens
    """
    # Check what we have
    print(f"\nAvailable tokens: {available_tokens}")
    
    # Try to map pools to available tokens
    pool_mapping = {}
    
    # Check each pool
    pools_to_check = [
        ('link_weth', ['LINK', 'ETH']),
        ('morpho_weth', ['MORPHO', 'ETH']),
        ('weth_inj', ['ETH', 'INJ']),
        ('wtao_weth', ['WTAO', 'ETH'])
    ]
    
    for pool_name, required_tokens in pools_to_check:
        available_for_pool = []
        missing_for_pool = []
        
        for token in required_tokens:
            if token in available_tokens:
                available_for_pool.append(token)
            else:
                missing_for_pool.append(token)
        
        if len(available_for_pool) == 2:
            pool_mapping[pool_name] = {
                'token1': available_for_pool[0],
                'token2': available_for_pool[1],
                'status': 'available'
            }
            print(f"  ✓ {pool_name}: {available_for_pool[0]} vs {available_for_pool[1]}")
        else:
            pool_mapping[pool_name] = {
                'token1': required_tokens[0],
                'token2': required_tokens[1],
                'status': 'missing',
                'missing': missing_for_pool
            }
            print(f"  ❌ {pool_name}: Missing {missing_for_pool}")
    
    return pool_mapping

# Get corrected pool configuration
corrected_pools = get_corrected_pool_config(available_tokens)


Available tokens: ['LINK', 'INJ', 'MORPHO', 'ETH', 'WTAO']
  ✓ link_weth: LINK vs ETH
  ✓ morpho_weth: MORPHO vs ETH
  ✓ weth_inj: ETH vs INJ
  ✓ wtao_weth: WTAO vs ETH


In [5]:
class PoolVolatilityCorrelationCalculator:
    """
    Calculates correlations and cross price volatility for Uniswap V3 pools
    Uses exponential weighting and hourly data
    """
    
    def __init__(self, lookback_days=90, half_life_days=30):
        """
        Args:
            lookback_days: Total lookback period (default: 90 days = 3 months)
            half_life_days: Half-life for exponential weighting (default: 30 days = 1 month)
        """
        self.lookback_days = lookback_days
        self.half_life_days = half_life_days
        
        # Calculate decay factor for exponential weighting
        self.decay_factor = np.log(2) / half_life_days
        
        print(f"Pool Volatility Correlation Calculator initialized:")
        print(f"  Lookback period: {lookback_days} days")
        print(f"  Half-life: {half_life_days} days")
        print(f"  Decay factor: {self.decay_factor:.6f}")
    
    def set_pool_configuration(self, pool_config):
        """
        Set pool configuration after checking available tokens
        """
        self.pools = pool_config
        print(f"  Pools to process: {list(self.pools.keys())}")
    
    def load_spot_data(self, data_dir='./data/centralized_prices/spot'):
        """
        Load all available spot price data and resample to hourly
        """
        print("Loading spot price data...")
        
        spot_data = {}
        if os.path.exists(data_dir):
            csv_files = glob.glob(os.path.join(data_dir, '*_binance_5m.csv'))
            
            for file in csv_files:
                token_name = os.path.basename(file).split('_')[0].upper()
                print(f"  Loading {token_name}...")
                
                try:
                    # Load 5-minute data
                    data_5min = pd.read_csv(file, index_col='datetime', parse_dates=True)
                    
                    if 'close' in data_5min.columns:
                        # Resample to hourly
                        data_hourly = data_5min.resample('H').agg({
                            'open': 'first',
                            'high': 'max',
                            'low': 'min',
                            'close': 'last',
                            'volume': 'sum'
                        }).dropna()
                        
                        spot_data[token_name] = data_hourly
                        print(f"    ✓ {token_name}: {len(data_5min)} 5-min → {len(data_hourly)} hourly records")
                    else:
                        print(f"    ❌ No 'close' column found in {token_name}")
                        
                except Exception as e:
                    print(f"    ❌ Error loading {token_name}: {e}")
        
        print(f"Loaded {len(spot_data)} token datasets")
        return spot_data
    
    def calculate_exponential_weighted_volatility(self, returns):
        """
        Calculate exponential weighted volatility
        """
        lookback_periods = self.lookback_days * 24  # 24 hours per day
        
        volatility = pd.Series(index=returns.index, dtype=float)
        
        for i in range(len(returns)):
            if i < lookback_periods:
                # Not enough data yet, use simple volatility
                volatility.iloc[i] = returns.iloc[:i+1].std()
            else:
                # Calculate exponentially weighted volatility
                start_idx = i - lookback_periods + 1
                window_returns = returns.iloc[start_idx:i+1]
                
                # Create exponential weights (more weight to recent data)
                weights = np.exp(-self.decay_factor * np.arange(len(window_returns))[::-1])
                weights = weights / weights.sum()  # Normalize weights
                
                # Calculate weighted volatility
                weighted_mean = np.average(window_returns, weights=weights)
                weighted_variance = np.average((window_returns - weighted_mean)**2, weights=weights)
                volatility.iloc[i] = np.sqrt(weighted_variance)
        
        # Annualize volatility (hourly to annual)
        annualization_factor = np.sqrt(24 * 365)
        volatility_annualized = volatility * annualization_factor
        
        return volatility_annualized
    
    def calculate_exponential_weighted_correlation(self, returns1, returns2):
        """
        Calculate exponential weighted correlation between two return series
        """
        lookback_periods = self.lookback_days * 24  # 24 hours per day
        
        correlation = pd.Series(index=returns1.index, dtype=float)
        
        for i in range(len(returns1)):
            if i < lookback_periods:
                correlation.iloc[i] = np.nan
            else:
                # Calculate exponentially weighted correlation
                start_idx = i - lookback_periods + 1
                window1 = returns1.iloc[start_idx:i+1]
                window2 = returns2.iloc[start_idx:i+1]
                
                # Create exponential weights
                weights = np.exp(-self.decay_factor * np.arange(len(window1))[::-1])
                weights = weights / weights.sum()
                
                # Calculate weighted correlation
                weighted_mean1 = np.average(window1, weights=weights)
                weighted_mean2 = np.average(window2, weights=weights)
                
                weighted_covariance = np.average((window1 - weighted_mean1) * (window2 - weighted_mean2), weights=weights)
                weighted_var1 = np.average((window1 - weighted_mean1)**2, weights=weights)
                weighted_var2 = np.average((window2 - weighted_mean2)**2, weights=weights)
                
                if weighted_var1 > 0 and weighted_var2 > 0:
                    correlation.iloc[i] = weighted_covariance / np.sqrt(weighted_var1 * weighted_var2)
                else:
                    correlation.iloc[i] = np.nan
        
        return correlation
    
    def calculate_cross_price_volatility(self, vol1, vol2, correlation):
        """
        Calculate cross price volatility using the formula:
        σ_cross = √(σ₁² + σ₂² - 2ρσ₁σ₂)
        """
        cross_vol = np.sqrt(vol1**2 + vol2**2 - 2 * correlation * vol1 * vol2)
        return cross_vol
    
    def process_all_pools(self, spot_data):
        """
        Process all pools and calculate correlations and cross price volatility
        """
        print("\nProcessing all pools...")
        print("=" * 60)
        
        pool_results = {}
        
        for pool_name, pool_info in self.pools.items():
            print(f"\nProcessing pool: {pool_name}")
            
            # Skip pools with missing tokens
            if pool_info.get('status') == 'missing':
                print(f"  ❌ Skipping {pool_name}: Missing tokens {pool_info.get('missing', [])}")
                continue
            
            print(f"  Tokens: {pool_info['token1']} vs {pool_info['token2']}")
            
            try:
                # Get token data
                token1_name = pool_info['token1']
                token2_name = pool_info['token2']
                
                if token1_name not in spot_data or token2_name not in spot_data:
                    print(f"  ❌ Missing data for {token1_name} or {token2_name}")
                    continue
                
                token1_data = spot_data[token1_name]
                token2_data = spot_data[token2_name]
                
                # Align data by timestamp
                common_index = token1_data.index.intersection(token2_data.index)
                if len(common_index) == 0:
                    print(f"  ❌ No common timestamps between {token1_name} and {token2_name}")
                    continue
                
                token1_aligned = token1_data.loc[common_index]
                token2_aligned = token2_data.loc[common_index]
                
                print(f"  Aligned data: {len(common_index)} common timestamps")
                
                # Calculate returns
                token1_returns = token1_aligned['close'].pct_change().dropna()
                token2_returns = token2_aligned['close'].pct_change().dropna()
                
                # Align returns
                common_returns_index = token1_returns.index.intersection(token2_returns.index)
                token1_returns = token1_returns.loc[common_returns_index]
                token2_returns = token2_returns.loc[common_returns_index]
                
                print(f"  Returns data: {len(common_returns_index)} common timestamps")
                
                # Calculate volatilities
                print(f"  Calculating volatilities...")
                token1_vol = self.calculate_exponential_weighted_volatility(token1_returns)
                token2_vol = self.calculate_exponential_weighted_volatility(token2_returns)
                
                # Calculate correlation
                print(f"  Calculating correlation...")
                correlation = self.calculate_exponential_weighted_correlation(token1_returns, token2_returns)
                
                # Calculate cross price volatility
                print(f"  Calculating cross price volatility...")
                cross_vol = self.calculate_cross_price_volatility(token1_vol, token2_vol, correlation)
                
                # Create result DataFrame
                result_df = pd.DataFrame({
                    'token1_price': token1_aligned.loc[common_returns_index, 'close'],
                    'token2_price': token2_aligned.loc[common_returns_index, 'close'],
                    'token1_returns': token1_returns,
                    'token2_returns': token2_returns,
                    'token1_volatility': token1_vol,
                    'token2_volatility': token2_vol,
                    'correlation': correlation,
                    'cross_price_volatility': cross_vol
                })
                
                # Add percentage columns
                result_df['token1_volatility_pct'] = result_df['token1_volatility'] * 100
                result_df['token2_volatility_pct'] = result_df['token2_volatility'] * 100
                result_df['cross_price_volatility_pct'] = result_df['cross_price_volatility'] * 100
                
                pool_results[pool_name] = result_df
                
                print(f"  ✓ Successfully processed {pool_name}")
                print(f"    Final shape: {result_df.shape}")
                print(f"    Volatility ranges:")
                print(f"      {token1_name}: {token1_vol.min():.2%} - {token1_vol.max():.2%}")
                print(f"      {token2_name}: {token2_vol.min():.2%} - {token2_vol.max():.2%}")
                print(f"      Cross price: {cross_vol.min():.2%} - {cross_vol.max():.2%}")
                print(f"    Correlation range: {correlation.min():.3f} - {correlation.max():.3f}")
                
            except Exception as e:
                print(f"  ❌ Error processing {pool_name}: {e}")
                import traceback
                traceback.print_exc()
        
        return pool_results
    
    def save_pool_results(self, pool_results, output_dir='./data/pool_volatility_correlation'):
        """
        Save results for each pool to individual CSV files
        """
        print(f"\nSaving pool results...")
        
        # Create output directory
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
            print(f"Created output directory: {output_dir}")
        
        saved_files = []
        
        for pool_name, result_df in pool_results.items():
            if result_df is not None and not result_df.empty:
                # Save to CSV
                output_file = os.path.join(output_dir, f"{pool_name}_volatility_correlation.csv")
                result_df.to_csv(output_file)
                
                saved_files.append(output_file)
                print(f"  ✓ Saved {pool_name}: {output_file}")
                print(f"    File size: {os.path.getsize(output_file) / 1024:.1f} KB")
        
        print(f"\nSaved {len(saved_files)} pool files")
        return saved_files
        

In [7]:
# Initialize the calculator
calculator = PoolVolatilityCorrelationCalculator(lookback_days=90, half_life_days=30)

# Set the corrected pool configuration
calculator.set_pool_configuration(corrected_pools)

# Load spot data
spot_data = calculator.load_spot_data()

# Process all pools
pool_results = calculator.process_all_pools(spot_data)

# Save results
saved_files = calculator.save_pool_results(pool_results)

Pool Volatility Correlation Calculator initialized:
  Lookback period: 90 days
  Half-life: 30 days
  Decay factor: 0.023105
  Pools to process: ['link_weth', 'morpho_weth', 'weth_inj', 'wtao_weth']
Loading spot price data...
  Loading LINK...
    ✓ LINK: 131230 5-min → 10936 hourly records
  Loading INJ...
    ✓ INJ: 131229 5-min → 10936 hourly records
  Loading MORPHO...
    ✓ MORPHO: 79479 5-min → 6624 hourly records
  Loading ETH...
    ✓ ETH: 131229 5-min → 10936 hourly records
  Loading WTAO...
    ✓ WTAO: 131230 5-min → 10936 hourly records
Loaded 5 token datasets

Processing all pools...

Processing pool: link_weth
  Tokens: LINK vs ETH
  Aligned data: 10936 common timestamps
  Returns data: 10935 common timestamps
  Calculating volatilities...
  Calculating correlation...
  Calculating cross price volatility...
  ✓ Successfully processed link_weth
    Final shape: (10935, 11)
    Volatility ranges:
      LINK: 19.17% - 246.98%
      ETH: 13.76% - 186.38%
      Cross price: 23.

In [8]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import os
import glob
import warnings
warnings.filterwarnings('ignore')

class MonthlyCorrelationCalculator:
    """
    Calculates monthly correlation matrices from hourly spot prices
    """
    
    def __init__(self):
        self.spot_data = {}
        self.hourly_data = {}
        
    def load_and_aggregate_spot_data(self, data_dir='./data/centralized_prices/spot'):
        """
        Load spot price data and aggregate to hourly (last value near each hour)
        """
        print("Loading and aggregating spot price data...")
        
        if os.path.exists(data_dir):
            csv_files = glob.glob(os.path.join(data_dir, '*_binance_5m.csv'))
            
            for file in csv_files:
                token_name = os.path.basename(file).split('_')[0].upper()
                print(f"  Processing {token_name}...")
                
                try:
                    # Load 5-minute data
                    data_5min = pd.read_csv(file, index_col='datetime', parse_dates=True)
                    
                    if 'close' in data_5min.columns:
                        # Aggregate to hourly, keeping last value near each hour
                        data_hourly = data_5min.resample('H').last().dropna()
                        
                        self.spot_data[token_name] = data_5min
                        self.hourly_data[token_name] = data_hourly
                        
                        print(f"    ✓ {token_name}: {len(data_5min)} 5-min → {len(data_hourly)} hourly records")
                        print(f"    Date range: {data_hourly.index.min()} to {data_hourly.index.max()}")
                    else:
                        print(f"    ❌ No 'close' column found in {token_name}")
                        
                except Exception as e:
                    print(f"    ❌ Error processing {token_name}: {e}")
        
        print(f"\nLoaded {len(self.hourly_data)} token datasets")
        return self.hourly_data
    
    def calculate_monthly_correlations(self, min_data_points=24*20):  # At least 20 days of hourly data
        """
        Calculate correlation matrix for each month with sufficient data
        """
        print("\nCalculating monthly correlation matrices...")
        print("=" * 60)
        
        if not self.hourly_data:
            print("❌ No hourly data loaded. Run load_and_aggregate_spot_data() first.")
            return {}
        
        # Get all available tokens
        tokens = list(self.hourly_data.keys())
        print(f"Available tokens: {tokens}")
        
        # Find common time range
        common_start = max([data.index.min() for data in self.hourly_data.values()])
        common_end = min([data.index.max() for data in self.hourly_data.values()])
        
        print(f"Common time range: {common_start} to {common_end}")
        
        # Create monthly periods
        monthly_periods = pd.date_range(start=common_start, end=common_end, freq='MS')  # Month Start
        
        monthly_correlations = {}
        
        for month_start in monthly_periods:
            month_end = month_start + pd.offsets.MonthEnd(1)
            month_name = month_start.strftime('%Y-%m')
            
            print(f"\nProcessing month: {month_name}")
            
            # Get data for this month
            month_data = {}
            valid_tokens = []
            
            for token in tokens:
                token_data = self.hourly_data[token]
                month_mask = (token_data.index >= month_start) & (token_data.index <= month_end)
                month_token_data = token_data[month_mask]
                
                if len(month_token_data) >= min_data_points:
                    month_data[token] = month_token_data['close']
                    valid_tokens.append(token)
                else:
                    print(f"  ⚠️  {token}: Only {len(month_token_data)} data points (need {min_data_points})")
            
            if len(valid_tokens) >= 2:
                # Create DataFrame for this month
                month_df = pd.DataFrame(month_data)
                
                # Calculate returns
                month_returns = month_df.pct_change().dropna()
                
                # Calculate correlation matrix
                correlation_matrix = month_returns.corr()
                
                monthly_correlations[month_name] = {
                    'correlation_matrix': correlation_matrix,
                    'data_points': len(month_returns),
                    'tokens': valid_tokens,
                    'start_date': month_start,
                    'end_date': month_end
                }
                
                print(f"  ✓ {month_name}: {len(month_returns)} data points, {len(valid_tokens)} tokens")
                
                # Show correlation matrix
                print(f"  Correlation Matrix:")
                print(correlation_matrix.round(3))
                
            else:
                print(f"  ❌ {month_name}: Insufficient data for correlation calculation")
        
        return monthly_correlations
    
    def display_correlation_summary(self, monthly_correlations):
        """
        Display summary of monthly correlations
        """
        print("\n" + "=" * 60)
        print("MONTHLY CORRELATION SUMMARY")
        print("=" * 60)
        
        if not monthly_correlations:
            print("No monthly correlations calculated.")
            return
        
        # Get all unique token pairs
        all_pairs = set()
        for month_data in monthly_correlations.values():
            corr_matrix = month_data['correlation_matrix']
            for i in range(len(corr_matrix.index)):
                for j in range(i+1, len(corr_matrix.columns)):
                    pair = f"{corr_matrix.index[i]}-{corr_matrix.columns[j]}"
                    all_pairs.add(pair)
        
        all_pairs = sorted(list(all_pairs))
        
        # Create summary DataFrame
        summary_data = []
        for month_name, month_data in monthly_correlations.items():
            row = {'Month': month_name, 'Data_Points': month_data['data_points']}
            
            corr_matrix = month_data['correlation_matrix']
            for pair in all_pairs:
                token1, token2 = pair.split('-')
                if token1 in corr_matrix.index and token2 in corr_matrix.columns:
                    row[pair] = corr_matrix.loc[token1, token2]
                else:
                    row[pair] = np.nan
            
            summary_data.append(row)
        
        summary_df = pd.DataFrame(summary_data)
        
        print("Monthly Correlation Summary:")
        print(summary_df.round(3))
        
        # Calculate average correlations across months
        print(f"\nAverage Correlations Across All Months:")
        for pair in all_pairs:
            values = summary_df[pair].dropna()
            if len(values) > 0:
                print(f"  {pair}: {values.mean():.3f} (std: {values.std():.3f})")
        
        return summary_df
    
    def save_monthly_correlations(self, monthly_correlations, output_dir='./data/monthly_correlations'):
        """
        Save monthly correlation data to CSV files
        """
        print(f"\nSaving monthly correlation data...")
        
        # Create output directory
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
            print(f"Created output directory: {output_dir}")
        
        saved_files = []
        
        for month_name, month_data in monthly_correlations.items():
            # Save correlation matrix
            corr_file = os.path.join(output_dir, f"{month_name}_correlation_matrix.csv")
            month_data['correlation_matrix'].to_csv(corr_file)
            
            # Save monthly summary
            summary_file = os.path.join(output_dir, f"{month_name}_summary.txt")
            with open(summary_file, 'w') as f:
                f.write(f"Month: {month_name}\n")
                f.write(f"Data Points: {month_data['data_points']}\n")
                f.write(f"Tokens: {', '.join(month_data['tokens'])}\n")
                f.write(f"Date Range: {month_data['start_date']} to {month_data['end_date']}\n\n")
                f.write("Correlation Matrix:\n")
                f.write(month_data['correlation_matrix'].to_string())
            
            saved_files.extend([corr_file, summary_file])
            print(f"  ✓ Saved {month_name}: {corr_file}, {summary_file}")
        
        print(f"\nSaved {len(saved_files)} files")
        return saved_files

In [9]:
# Initialize the calculator
monthly_calc = MonthlyCorrelationCalculator()

# Load and aggregate spot data
hourly_data = monthly_calc.load_and_aggregate_spot_data()

# Calculate monthly correlations
monthly_correlations = monthly_calc.calculate_monthly_correlations(min_data_points=24*20)  # At least 20 days

# Display summary
summary_df = monthly_calc.display_correlation_summary(monthly_correlations)

# Save results
saved_files = monthly_calc.save_monthly_correlations(monthly_correlations)

Loading and aggregating spot price data...
  Processing LINK...
    ✓ LINK: 131230 5-min → 8777 hourly records
    Date range: 2024-08-29 21:00:00 to 2025-08-30 13:00:00
  Processing INJ...
    ✓ INJ: 131229 5-min → 8777 hourly records
    Date range: 2024-08-29 21:00:00 to 2025-08-30 13:00:00
  Processing MORPHO...
    ✓ MORPHO: 79479 5-min → 4464 hourly records
    Date range: 2025-02-25 14:00:00 to 2025-08-30 13:00:00
  Processing ETH...
    ✓ ETH: 131229 5-min → 8777 hourly records
    Date range: 2024-08-29 21:00:00 to 2025-08-30 13:00:00
  Processing WTAO...
    ✓ WTAO: 131230 5-min → 8777 hourly records
    Date range: 2024-08-29 21:00:00 to 2025-08-30 13:00:00

Loaded 5 token datasets

Calculating monthly correlation matrices...
Available tokens: ['LINK', 'INJ', 'MORPHO', 'ETH', 'WTAO']
Common time range: 2025-02-25 14:00:00 to 2025-08-30 13:00:00

Processing month: 2025-03
  ✓ 2025-03: 720 data points, 5 tokens
  Correlation Matrix:
         LINK    INJ  MORPHO    ETH   WTAO
L